# 05 — Layer Normalization

**Ladder:** Multi-pass stats → fused per-row (Python) → masked power-of-two block → Triton forward (+ save $\mu$, $r_{\mathrm{std}}$) → **full backward derivation**.

**Study tip.** Fuse mean/var/affine so $x$ is not re-read from HBM between passes. Know how RMSNorm drops the mean. The backward pass is derived from scratch in §4; it is the mathematical heart of this folder.

Official: [Layer Norm](https://triton-lang.org/main/getting-started/tutorials/05-layer-norm.html)

Figures: [`docs/images/`](docs/images/) · deep dive: [`docs/README.md`](docs/README.md)

---

# Part I — Mathematics

## 0. Forward pass: definition and identities

Work on one row first; the matrix case applies the same map independently to
each row. Fix $x\in\mathbb{R}^{N}$, weights $\gamma,\beta\in\mathbb{R}^{N}$, and
$\varepsilon>0$.

**Definition 1 (layer normalization).** Define the mean, variance, standard
deviation, normalized row, and output by

$$
\begin{aligned}
\mu &= \frac{1}{N}\sum_{j=0}^{N-1}x_j,\\[2pt]
\sigma^{2} &= \frac{1}{N}\sum_{j=0}^{N-1}(x_j-\mu)^{2},\\[2pt]
s &= \sqrt{\sigma^{2}+\varepsilon},\\[2pt]
\hat x_j &= \frac{x_j-\mu}{s},\\[2pt]
y_j &= \gamma_j\,\hat x_j+\beta_j .
\end{aligned}
\tag{0.1}
$$

For $X\in\mathbb{R}^{M\times N}$ the same formulas are applied to each row
$X_{r,:}$, producing $Y_{r,:}$.

**Proposition 1 (zero mean, unit variance).** The normalized row satisfies

$$
\frac{1}{N}\sum_{j=0}^{N-1}\hat x_j=0,
\qquad
\frac{1}{N}\sum_{j=0}^{N-1}\hat x_j^{2}
=
\frac{\sigma^{2}}{\sigma^{2}+\varepsilon}
=
1-\frac{\varepsilon}{\sigma^{2}+\varepsilon}
\;\le\;1.
\tag{0.2}
$$

*Proof.* The first identity follows from
$\sum_j(x_j-\mu)=\sum_jx_j-N\mu=0$. For the second, use the definition of
$\sigma^{2}$:

$$
\frac1N\sum_j\hat x_j^{2}
=
\frac1N\sum_j\frac{(x_j-\mu)^{2}}{s^{2}}
=
\frac{\sigma^{2}}{\sigma^{2}+\varepsilon}.
\qquad\square
$$

So layer norm produces exactly zero mean and variance $1$ in the
$\varepsilon\to0$ limit. The bias $\varepsilon$ shrinks the variance slightly,
which is the price of avoiding division by zero.

### 0.1 Affine equivariance (centering)

**Proposition 2.** For every $c\in\mathbb{R}$,

$$
\mathrm{LN}(x+c\mathbf{1})=\mathrm{LN}(x).
\tag{0.3}
$$

*Proof.* The shifted mean is $\mu'=\mu+c$, so
$x_j+c-\mu'=x_j-\mu$ and
$\sigma'^{2}=\frac1N\sum_j(x_j-\mu)^{2}=\sigma^{2}$. Hence $\hat x'=\hat x$.
$\square$

This is why layer norm is insensitive to a constant offset but **not** to a
rescaling: $\mathrm{LN}(ax)=a\,\mathrm{LN}(x)$ only for $a>0$ when
$\varepsilon=0$ (for $a<0$ the sign flips through the mean), which is exactly the
property that motivates the learnable $\gamma$.

### 0.2 The one-pass variance identity

Expanding the square,

$$
\begin{aligned}
\sigma^{2}
&=\frac1N\sum_j(x_j-\mu)^{2}\\
&=\frac1N\sum_jx_j^{2}-\frac{2\mu}{N}\sum_jx_j+\mu^{2}\\
&=\frac1N\sum_jx_j^{2}-2\mu^{2}+\mu^{2}
=\frac1N\sum_jx_j^{2}-\mu^{2}.
\end{aligned}
\tag{0.4}
$$

This lets a kernel compute $\mu$ and $\sum_jx_j^{2}$ in one pass. The catch is
**catastrophic cancellation**: if $\mu^{2}\gg\sigma^{2}$ (data far from zero and
tightly clustered), the subtraction loses significant digits. The centered form
$\frac1N\sum_j(x_j-\mu)^{2}$ used in this notebook needs two passes over the row
but is numerically better behaved. For rows that fit in SRAM (the usual case for
layer norm), the second pass is free.

**Error bound.** In floating point, evaluating (0.4) gives a relative error of
order $u\,\mu^{2}/\sigma^{2}$, which can be enormous when the ratio is large.
The centered form has relative error $O(Nu)$ with a much smaller constant. This
is the precise reason Triton tutorials use the centered form in FP32.

### 0.3 Worked example

Take $x=(1,2,3,4)$, $\gamma=\mathbf{1}$, $\beta=\mathbf{0}$. Then

$$
\mu=\frac{10}{4}=2.5,
\qquad
\sigma^{2}=\frac{(-1.5)^{2}+(-0.5)^{2}+0.5^{2}+1.5^{2}}{4}
=\frac{5}{4}=1.25,
$$

$$
s=\sqrt{1.25+\varepsilon}\approx1.1180,
\qquad
\hat x\approx(-1.3416,\,-0.4472,\,0.4472,\,1.3416).
$$

Check: $\sum_j\hat x_j=0$ and $\frac14\sum_j\hat x_j^{2}=1$ (for
$\varepsilon=0$).

### 0.4 RMSNorm: drop the mean

**Definition 2 (RMSNorm).** With the root-mean-square statistic

$$
\mathrm{RMS}(x)=\sqrt{\frac1N\sum_{j=0}^{N-1}x_j^{2}+\varepsilon},
\tag{0.5}
$$

RMSNorm is

$$
y_j=\gamma_j\,\frac{x_j}{\mathrm{RMS}(x)}.
\tag{0.6}
$$

It has no mean subtraction and no bias $\beta$, hence one fewer reduction and
one fewer parameter vector. It is not shift-invariant (adding a constant changes
$\mathrm{RMS}$), which is acceptable in architectures that already center
activations (e.g. after a residual stream). The backward pass is derived in
§4.11.

### 0.5 HBM pass count

| Implementation | Reads of $x$ | Writes |
|---|---|---|
| Multi-pass PyTorch | 3 (mean, var, normalize) | $y$ (plus intermediates) |
| Fused row program | 1 | $y$ and two scalars |
| Triton forward | 1 load of the row | $y$, $\mu$, $r_{\mathrm{std}}$ |

Fusion turns three sweeps into one. Since layer norm is memory-bound
($\mathrm{AI}\approx$ a few FLOP/byte), this is the dominant optimization; the
reductions themselves are cheap.

![Whiteboard: multi-pass layer-norm vs fused row kernel](docs/images/benchmark-layer-norm.png)

*Theory board — mean/var/affine as separate HBM passes vs one fused load/reduce/store.*

### Theory board

![Whiteboard: multi-pass layer-norm vs fused row kernel](docs/images/whiteboard-theory.jpg)

*Theory board — mean/var/affine as separate HBM passes vs one fused load/reduce/store.*

# Part II — Proof

**Theorem (forward normalization).** With $\mu,\sigma^2,s,\hat x_j$ as in
(0.1), the normalized row satisfies $\sum_j\hat x_j=0$ and
$\frac1N\sum_j\hat x_j^2=\sigma^2/(\sigma^2+\varepsilon)$.

*Proof.* $\sum_j(x_j-\mu)=\sum_jx_j-N\mu=0$. For the second moment,
$\sum_j(x_j-\mu)^2=N\sigma^2$, so dividing by $Ns^2$ gives
$\sigma^2/(\sigma^2+\varepsilon)$. $\square$

**Theorem (backward formula).** Let $g_j=\partial L/\partial y_j$,
$d\hat z_j=g_j\gamma_j$, and let $\overline{d\hat z}$,
$\overline{d\hat z\odot\hat x}$ be the feature means. Then

$$
\frac{\partial L}{\partial x_i}
=
\frac1s\left[
d\hat z_i-\overline{d\hat z}
-\hat x_i\,\overline{d\hat z\odot\hat x}
\right].
$$

*Proof.* The Jacobian is

$$
\frac{\partial\hat x_j}{\partial x_i}
=
\frac1s\left[\delta_{ij}-\frac1N-\frac{\hat x_j\hat x_i}{N}\right]
$$

(derive from $\partial\mu/\partial x_i=1/N$,
$\partial\sigma^2/\partial x_i=2(x_i-\mu)/N$,
$\partial s/\partial x_i=(x_i-\mu)/(Ns)$). Substituting into
$\partial L/\partial x_i=\sum_jd\hat z_j\,\partial\hat x_j/\partial x_i$
and collecting the two means gives the formula. $\square$

**Corollary (parameter gradients).**
$\partial L/\partial\beta_j=\sum_rg_{rj}$ and
$\partial L/\partial\gamma_j=\sum_rg_{rj}\hat x_{rj}$.

*Proof.* Differentiate $y_j=\gamma_j\hat x_j+\beta_j$ with respect to the
parameters and sum over rows. $\square$

# Part III — Step-by-step proof

**Forward**

1. Load the row and cast to FP32.
2. Compute $\mu=\frac1N\sum_jx_j$.
3. Center: $u_j=x_j-\mu$.
4. Compute $\sigma^2=\frac1N\sum_ju_j^2$.
5. Compute $r=1/\sqrt{\sigma^2+\varepsilon}$.
6. Normalize $\hat x_j=u_jr$ and apply $y_j=\gamma_j\hat x_j+\beta_j$.
7. Store $y$, and save $\mu,r$ for backward.

**Backward**

1. Load $x,dY,\mu,r,\gamma$.
2. Recompute $\hat x=(x-\mu)r$.
3. Form $d\hat z=dY\odot\gamma$.
4. Reduce $m_1=\operatorname{mean}(d\hat z\odot\hat x)$ over features.
5. Reduce $m_2=\operatorname{mean}(d\hat z)$ over features.
6. Form $dx=r\,[d\hat z-m_1\hat x-m_2]$.
7. Accumulate $d\gamma=\sum_rdY\odot\hat x$,
   $d\beta=\sum_rdY$ over rows.
8. Reduce partial sums instead of atomically adding to avoid contention.

# Part IV — Coding


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



### Method 01 — Naive (multi-pass)

**Naive.** Three logical passes over $x$: mean, variance, affine normalize
(`naive_layer_norm_pseudo`).

**Fused (still exact).** One loop body per row $r$
(`fused_layer_norm_pseudo`):

$$
\mu_r\;\longrightarrow\;\sigma_r^{2}\;\longrightarrow\;y_{r,:}
\tag{2.1}
$$

with all temporaries local. Same numerics as the multi-pass form; fewer
conceptual DRAM sweeps.

### 2.1 Traffic

For one row of length $N$ with element size $b$, the naive chain performs

$$
\underbrace{N}_{\text{mean: read }x}
+\underbrace{N}_{\text{var: read }x}
+\underbrace{N}_{\text{normalize: read }x}
+\underbrace{N}_{\text{write }y}
=4N
\ \text{element transfers}.
\tag{2.2}
$$

The fused row program performs

$$
\underbrace{N}_{\text{load }x}
+\underbrace{N}_{\text{store }y}
=2N
\tag{2.3}
$$

element transfers, plus two scalars ($\mu$ and $r_{\mathrm{std}}$). Thus

$$
\frac{T_{\mathrm{naive}}}{T_{\mathrm{fused}}}
\approx 2.
\tag{2.4}
$$

If $\gamma,\beta$ are also loaded, the fused traffic becomes $4N$ per row
($x,\gamma,\beta,y$). Note that $\gamma,\beta$ are shared across rows and are
therefore likely L2-resident; the real HBM cost is still close to $2N$.

### 2.2 Precision

The statistics are accumulations over the whole row, so their error grows with
$N$. The implementation upcasts each element to `float32` inside the kernel:

$$
x_j^{\mathrm{fp32}}=\operatorname{cast}(x_j),
\qquad
\mu=\frac1N\sum_jx_j^{\mathrm{fp32}},
\tag{2.5}
$$

and performs the variance sum in FP32 as well. This avoids the large rounding
error of FP16 accumulation while never materializing an FP32 copy of the tensor.
The final output is cast back to the input dtype.

**Storage of statistics.** $\mu$ and $r_{\mathrm{std}}=1/s$ are saved as FP32
even for FP16 inputs. The backward formula (4.8) multiplies by $r_{\mathrm{std}}$
and by $1/N$; storing either in FP16 would inject a relative error of
$2^{-11}\approx5\times10^{-4}$ into every gradient component.

### 2.3 Teaching sizes

$M=4$, $N\in\{8,16,255\}$ — the odd $N$ exercises the mask path before scaling
to $1024+$. For $N=255$ the block size is $B=256$ and exactly one lane is
masked; that single lane is enough to catch a missing mask in a test.


In [ ]:
def naive_layer_norm_pseudo(x, weight, bias, eps=1e-5):
    """
    Separate passes over each row — easy to read, poor memory traffic.

    Pass 1: mean
    Pass 2: variance
    Pass 3: normalize + affine
    """
    M, N = x.shape
    y = torch.empty_like(x)
    for m in range(M):
        row = x[m]
        mean = row.sum() / N
        var = ((row - mean) ** 2).sum() / N
        y[m] = (row - mean) / torch.sqrt(var + eps) * weight + bias
    return y


### Method 02 — Kernel-based (forward only)

With $\mathrm{pid}=r$ (one row) and block size
$B=\mathrm{next\_power\_of\_2}(N)$:

$$
\mu\leftarrow\frac1N\sum_jx_j,
\qquad
r_{\mathrm{std}}\leftarrow\big(\sigma^{2}+\varepsilon\big)^{-1/2},
\qquad
y_j\leftarrow(x_j-\mu)\,r_{\mathrm{std}}\,\gamma_j+\beta_j .
\tag{3.1}
$$

The kernel stores $\mu$ and $r_{\mathrm{std}}$ for the backward pass. The mask
$j<N$ on a power-of-two block implements partial rows safely.

### 3.1 Masked reduction

The row is loaded as

$$
\tilde x_j=
\begin{cases}
x_j, & j<N,\\
0, & N\le j<B.
\end{cases}
\tag{3.2}
$$

Using `other=0` is correct for the **sum** (the neutral element of addition), so

$$
\sum_{j=0}^{B-1}\tilde x_j=\sum_{j=0}^{N-1}x_j.
\tag{3.3}
$$

But it is not correct for the centered second moment unless the padded lanes are
explicitly zeroed after centering. The kernel therefore computes

$$
\tilde x_j^{\,c}
=
\begin{cases}
\tilde x_j-\mu, & j<N,\\
0, & N\le j<B,
\end{cases}
\qquad
\sigma^{2}=\frac1N\sum_{j=0}^{B-1}\big(\tilde x_j^{\,c}\big)^{2}.
\tag{3.4}
$$

Without the `tl.where(mask, ..., 0)` step, each padded lane would contribute
$\mu^{2}$ to the variance and the result would be wrong by
$(B-N)\mu^{2}/N$. This is the single most common layer-norm kernel bug.

### 3.2 Per-row cost

$$
\text{loads}=N\ (x)+N\ (\gamma)+N\ (\beta),
\qquad
\text{stores}=N\ (y)+2,
\qquad
\text{FLOPs}\approx 5N.
\tag{3.5}
$$

The arithmetic intensity is $\approx5/(4b)$ FLOP/byte for `float32`; the kernel
is memory-bound. Its runtime is approximately

$$
t_r\approx\frac{(3N+2)b}{\mathrm{BW}},
\qquad
t_{\mathrm{total}}\approx\frac{M(3N+2)b}{\mathrm{BW}}
\tag{3.6}
$$

when rows are large enough to amortize the launch.

| Stage | Function | DRAM sweeps of $x$ |
|-------|----------|--------------------|
| Naive | `naive_layer_norm_pseudo` | multi-pass |
| Fused Python | `fused_layer_norm_pseudo` | fused locally |
| Triton | `_layer_norm_fwd_kernel` / `layer_norm` | one load + store $y$ |

### 3.3 Cousin: RMSNorm

RMSNorm drops the mean:

$$
y_j=\gamma_j\,\frac{x_j}{\sqrt{\frac1N\sum_kx_k^{2}+\varepsilon}}.
\tag{3.7}
$$

Same fusion pattern, one fewer reduction, no $\beta$. The backward pass is
derived in §4.11.


In [ ]:
@triton.jit
def _layer_norm_fwd_kernel(
    Y,
    X,
    W,
    B,
    Mean,
    Rstd,
    stride,
    N,
    eps,
    BLOCK_SIZE: tl.constexpr,
):
    # One program per row — same ownership as fused_layer_norm_pseudo
    row = tl.program_id(0)
    Y += row * stride
    X += row * stride
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < N
    x = tl.load(X + cols, mask=mask, other=0.0).to(tl.float32)
    mean = tl.sum(x, axis=0) / N
    x_centered = tl.where(mask, x - mean, 0.0)
    var = tl.sum(x_centered * x_centered, axis=0) / N
    rstd = 1 / tl.sqrt(var + eps)
    w = tl.load(W + cols, mask=mask, other=0.0).to(tl.float32)
    b = tl.load(B + cols, mask=mask, other=0.0).to(tl.float32)
    y = (x - mean) * rstd * w + b
    tl.store(Mean + row, mean)
    tl.store(Rstd + row, rstd)
    tl.store(Y + cols, y, mask=mask)


def layer_norm(x, weight, bias, eps=1e-5):
    M, N = x.shape
    x = x.contiguous()          # the kernel assumes a dense trailing dimension
    y = torch.empty_like(x)
    mean = torch.empty((M,), device=x.device, dtype=torch.float32)
    rstd = torch.empty((M,), device=x.device, dtype=torch.float32)
    BLOCK_SIZE = triton.next_power_of_2(N)
    _layer_norm_fwd_kernel[(M,)](
        y,
        x,
        weight,
        bias,
        mean,
        rstd,
        x.stride(0),
        N,
        eps,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return y


### Method 03 — Fused kernel-based (forward + backward)

Method 03 keeps the one-row-per-program forward kernel and adds the backward
kernel derived in Part II. The backward pass tiles rows, recomputes
$\hat x=(x-\mu)r$, applies the boxed formula for $dx$, and writes **partial**
sums of $d\gamma,d\beta$ to a $(P\times E)$ buffer; a final column reduction
replaces contended atomics.


In [ ]:
def fused_layer_norm_pseudo(x, weight, bias, eps=1e-5):
    """
    Still Python, but one function per row — mirrors the fused Triton program
    that owns a single row (pid = row index).
    """
    M, N = x.shape
    y = torch.empty_like(x)
    for m in range(M):
        row = x[m].to(torch.float32)
        mean = row.sum() / N
        centered = row - mean
        var = (centered * centered).sum() / N
        rstd = 1.0 / torch.sqrt(var + eps)
        y[m] = (centered * rstd * weight + bias).to(x.dtype)
    return y


#### Backward derivation (matching the PDF)

This section reproduces, step by step, the handwritten derivation in
[`docs/LayerNorm_kernal .pdf`](docs/LayerNorm_kernal%20.pdf). We keep the row
index fixed and drop it; all formulas are then applied independently to every
row of $X$.

### 4.1 Setup and notation

Let $L$ be a scalar loss and let

$$
g_j:=\frac{\partial L}{\partial y_j},
\qquad j\in[N],
\tag{4.1}
$$

be the **upstream gradient** of the output. The forward map is

$$
y_j=\gamma_j\hat x_j+\beta_j,
\qquad
\hat x_j=\frac{x_j-\mu}{s},
\qquad
s=\sqrt{\sigma^{2}+\varepsilon},
\tag{4.2}
$$

with

$$
\mu=\frac1N\sum_{k=0}^{N-1}x_k,
\qquad
\sigma^{2}=\frac1N\sum_{k=0}^{N-1}(x_k-\mu)^{2}.
\tag{4.3}
$$

**Goal.** Compute

$$
\frac{\partial L}{\partial x_i},
\qquad
\frac{\partial L}{\partial \gamma_j},
\qquad
\frac{\partial L}{\partial \beta_j}
\qquad
\text{for all }i,j\in[N].
\tag{4.4}
$$

### 4.2 Gradients with respect to the affine parameters

Because $y_j$ depends on $\beta_j$ only through the additive term, and on
$\gamma_j$ only through the multiplicative term,

$$
\frac{\partial y_j}{\partial\beta_j}=1,
\qquad
\frac{\partial y_j}{\partial\gamma_j}=\hat x_j,
\qquad
\frac{\partial y_j}{\partial\beta_k}=\frac{\partial y_j}{\partial\gamma_k}=0
\quad(k\ne j).
\tag{4.5}
$$

Hence, for one row,

$$
\boxed{\;
\frac{\partial L}{\partial\beta_j}=g_j,
\qquad
\frac{\partial L}{\partial\gamma_j}=g_j\,\hat x_j .
\;}
\tag{4.6}
$$

For a batch $X\in\mathbb{R}^{M\times N}$ the parameters are shared across rows,
so the per-row contributions sum:

$$
\boxed{\;
\frac{\partial L}{\partial\beta_j}
=
\sum_{r=0}^{M-1}g_{rj},
\qquad
\frac{\partial L}{\partial\gamma_j}
=
\sum_{r=0}^{M-1}g_{rj}\,\hat x_{rj}.
\;}
\tag{4.7}
$$

This is a **column-wise reduction** of the $M\times N$ arrays $g$ and
$g\odot\hat x$: sum over rows, keep the feature index. It is the reason the
backward kernel needs a reduction along the batch axis, not the feature axis.

### 4.3 Chain rule down to $x$

Since $y_j$ depends on $x_i$ only through $\hat x_j$,

$$
\frac{\partial L}{\partial x_i}
=
\sum_{j=0}^{N-1}
\frac{\partial L}{\partial y_j}
\frac{\partial y_j}{\partial \hat x_j}
\frac{\partial \hat x_j}{\partial x_i}
=
\sum_{j=0}^{N-1}g_j\gamma_j\,
\frac{\partial \hat x_j}{\partial x_i}.
\tag{4.8}
$$

Define the **upstream gradient with respect to $\hat x$**:

$$
d\hat z_j:=g_j\gamma_j=\frac{\partial L}{\partial\hat x_j}.
\tag{4.9}
$$

Then

$$
\frac{\partial L}{\partial x_i}
=
\sum_{j=0}^{N-1}d\hat z_j\,
\frac{\partial \hat x_j}{\partial x_i}.
\tag{4.10}
$$

The only nontrivial object is the Jacobian
$J_{ji}=\partial\hat x_j/\partial x_i$. We compute it in three steps.

#### Step 1: derivative of the mean

$$
\mu=\frac1N\sum_{k=0}^{N-1}x_k
\quad\Longrightarrow\quad
\frac{\partial\mu}{\partial x_i}=\frac1N .
\tag{4.11}
$$

#### Step 2: derivative of the variance

Let $u_k=x_k-\mu$. Then $\partial u_k/\partial x_i=\delta_{ki}-1/N$, and

$$
\begin{aligned}
\frac{\partial\sigma^{2}}{\partial x_i}
&=
\frac1N\sum_{k=0}^{N-1}2u_k\frac{\partial u_k}{\partial x_i}\\
&=
\frac{2}{N}\sum_{k=0}^{N-1}u_k\left(\delta_{ki}-\frac1N\right)\\
&=
\frac{2}{N}\left(u_i-\frac1N\sum_{k=0}^{N-1}u_k\right).
\end{aligned}
\tag{4.12}
$$

But $\sum_k u_k=\sum_k(x_k-\mu)=0$, so

$$
\boxed{\;
\frac{\partial\sigma^{2}}{\partial x_i}
=
\frac{2}{N}\,(x_i-\mu)
=
\frac{2}{N}\,u_i .
\;}
\tag{4.13}
$$

This is the key cancellation: the mean-centering makes the average of $u$ zero,
so only the diagonal term survives. Without it, the derivative would contain a
spurious average of all coordinates.

#### Step 3: derivative of the standard deviation

Since $s=(\sigma^{2}+\varepsilon)^{1/2}$,

$$
\frac{\partial s}{\partial x_i}
=
\frac{1}{2s}\frac{\partial\sigma^{2}}{\partial x_i}
=
\frac{1}{2s}\cdot\frac{2(x_i-\mu)}{N}
=
\frac{x_i-\mu}{Ns}.
\tag{4.14}
$$

#### Step 4: assemble the Jacobian

Write $\hat x_j=u_j/s$. By the product/quotient rule,

$$
\begin{aligned}
\frac{\partial\hat x_j}{\partial x_i}
&=
\frac{1}{s}\frac{\partial u_j}{\partial x_i}
+
u_j\frac{\partial}{\partial x_i}\left(\frac1s\right)\\
&=
\frac{1}{s}\left(\delta_{ij}-\frac1N\right)
-
\frac{u_j}{s^{2}}\frac{\partial s}{\partial x_i}\\
&=
\frac{1}{s}\left(\delta_{ij}-\frac1N\right)
-
\frac{u_j}{s^{2}}\cdot\frac{u_i}{Ns}.
\end{aligned}
\tag{4.15}
$$

Using $u_j/s=\hat x_j$ and $u_i/s=\hat x_i$, the last term is
$\hat x_j\hat x_i/(Ns)$, so

$$
\boxed{\;
J_{ji}
=
\frac{\partial\hat x_j}{\partial x_i}
=
\frac{1}{s}\left[
\delta_{ij}-\frac1N-\frac{\hat x_j\hat x_i}{N}
\right].
\;}
\tag{4.16}
$$

**Corollary.** $\sum_j J_{ji}=0$ and $\sum_i J_{ji}=0$: the Jacobian annihilates
the constant direction, consistent with shift invariance (Proposition 2).

*Proof.* Sum (4.16) over $j$: $\sum_j\delta_{ij}=1$, $\sum_j1/N=1$, and
$\sum_j\hat x_j=0$ by Proposition 1. The same argument for the sum over $i$. $\square$

### 4.4 The final gradient with respect to $x$

Substitute (4.16) into (4.10):

$$
\begin{aligned}
\frac{\partial L}{\partial x_i}
&=
\frac{1}{s}\sum_{j=0}^{N-1}d\hat z_j
\left[
\delta_{ij}-\frac1N-\frac{\hat x_j\hat x_i}{N}
\right]\\
&=
\frac{1}{s}\left[
d\hat z_i
-\frac1N\sum_{j=0}^{N-1}d\hat z_j
-\frac{\hat x_i}{N}\sum_{j=0}^{N-1}d\hat z_j\hat x_j
\right].
\end{aligned}
\tag{4.17}
$$

Define the two scalar reductions

$$
\overline{d\hat z}:=\frac1N\sum_{j=0}^{N-1}d\hat z_j,
\qquad
\overline{d\hat z\odot\hat x}:=\frac1N\sum_{j=0}^{N-1}d\hat z_j\hat x_j .
\tag{4.18}
$$

Then the boxed result is

$$
\boxed{\;
\frac{\partial L}{\partial x_i}
=
\frac{1}{s}\left[
d\hat z_i
-\overline{d\hat z}
-\hat x_i\,\overline{d\hat z\odot\hat x}
\right],
\qquad
s=\sqrt{\sigma^{2}+\varepsilon}.
\;}
\tag{4.19}
$$

This is exactly the formula in the PDF:

$$
\frac{\partial L}{\partial x}
=
\frac{1}{\sqrt{\sigma^{2}+\varepsilon}}
\left[
d\hat z-\operatorname{mean}(d\hat z)
-\hat x\odot\operatorname{mean}(d\hat z\odot\hat x)
\right].
\tag{4.20}
$$

**Interpretation of the three terms.**

1. $d\hat z_i$ — direct sensitivity of the output to this coordinate.
2. $-\overline{d\hat z}$ — removes the component that changes the mean; a
   uniform shift of all coordinates must not change the normalized output.
3. $-\hat x_i\,\overline{d\hat z\odot\hat x}$ — removes the component that
   changes the variance; scaling all coordinates must not change the normalized
   output either.

Together they project $d\hat z$ onto the subspace orthogonal to
$\operatorname{span}\{\mathbf{1},\hat x\}$, which is precisely the tangent space
of the normalization constraint.

### 4.5 Worked numerical example

Take $x=(1,2,3,4)$, $\gamma=\mathbf{1}$, $\beta=\mathbf{0}$,
$\varepsilon=0$, and the simplest upstream gradient $g=(1,1,1,1)$. From §0.3,

$$
\mu=2.5,\quad
\sigma^{2}=1.25,\quad
s\approx1.118034,\quad
\hat x\approx(-1.34164,-0.44721,0.44721,1.34164).
$$

Here $d\hat z=g$, so

$$
\overline{d\hat z}=1,
\qquad
\overline{d\hat z\odot\hat x}
=
\frac{1}{4}\sum_j\hat x_j
=0.
$$

Therefore (4.19) gives

$$
\frac{\partial L}{\partial x_i}
=
\frac{1}{s}\big(1-1-\hat x_i\cdot0\big)
=0
\qquad\text{for all }i.
$$

**Sanity check.** $\sum_j y_j=\sum_j\hat x_j=0$ identically in $x$, so the
gradient of $L=\sum_j y_j$ with respect to every $x_i$ must vanish. The formula
reproduces this exactly.

A less trivial case is $g=(1,0,0,0)$ (only $y_0$ enters the loss). Then
$d\hat z=g$, $\overline{d\hat z}=1/4$, and
$\overline{d\hat z\odot\hat x}=\hat x_0/4\approx-0.33541$. Equation (4.19)
yields

$$
\frac{\partial L}{\partial x}
\approx
(0.2683,\,-0.3578,\,-0.0894,\,0.1789),
\qquad
\sum_i\frac{\partial L}{\partial x_i}=0,
$$

which again respects the shift-invariance constraint.

For the parameters, (4.6) gives

$$
\frac{\partial L}{\partial\gamma}
=
\hat x
\approx(-1.34164,-0.44721,0.44721,1.34164),
\qquad
\frac{\partial L}{\partial\beta}
=
(1,1,1,1).
$$

### 4.6 Row-batched form used by the kernel

For a tile of $R$ rows, let $X\in\mathbb{R}^{R\times N}$, $dY\in\mathbb{R}^{R\times N}$,
and let $\mu_r,r_r$ be the saved forward statistics. Then, with all operations
row-wise,

$$
\begin{aligned}
\hat X_{rj}
&=
(X_{rj}-\mu_r)\,r_r,\\
d\hat Z_{rj}
&=
dY_{rj}\,\gamma_j,\\
m^{(1)}_r
&=
\frac1N\sum_{j=0}^{N-1}d\hat Z_{rj}\,\hat X_{rj},\\
m^{(2)}_r
&=
\frac1N\sum_{j=0}^{N-1}d\hat Z_{rj},\\
dX_{rj}
&=
r_r\left(
d\hat Z_{rj}
-
m^{(1)}_r\hat X_{rj}
-
m^{(2)}_r
\right).
\end{aligned}
\tag{4.21}
$$

This is precisely the code in the reference backward kernel:

```python
x_hat  = (x - row_mean) * row_inv_std
wdy    = gamma * dy
mean1  = tl.sum(x_hat * wdy, axis=1)[:, None] / n_cols
mean2  = tl.sum(wdy, axis=1)[:, None] / n_cols
dx     = row_inv_std * (wdy - (x_hat * mean1 + mean2))
```

**Note on storage.** The forward saves $\mu$ and $r=1/s$ (two scalars per row)
rather than $\hat X$ (an $M\times N$ tensor). Recomputing $\hat X$ in the
backward costs $2N$ FLOPs per row but saves $MNb$ bytes of HBM traffic — a
classic recompute-versus-store trade.

### 4.7 The atomic-add problem for $d\gamma,d\beta$

From (4.7), the parameter gradients are sums over **all** rows. A natural
implementation is: each program, after processing its rows, executes

```python
tl.atomic_add(dgamma_ptr + cols, partial_dgamma)
tl.atomic_add(dbeta_ptr  + cols, partial_dbeta)
```

with $P$ programs and $E=N$ features. The cost is

$$
\#\text{atomics}=2PE,
\tag{4.22}
$$

and every atomic to the same address is serialized. If $P\approx M/R$ is large
(thousands of programs) and $E$ is large (thousands of features), the atomics
dominate the runtime: column $j$ is touched $P$ times in sequence, and the
contention grows with the number of programs.

**Two-phase reduction.** Instead, allocate a partial-sum buffer

$$
D_\gamma\in\mathbb{R}^{P\times E},
\qquad
D_\beta\in\mathbb{R}^{P\times E},
\tag{4.23}
$$

let program $p$ **store** (not atomically add) its partial sums to row $p$, and
reduce afterwards:

$$
\frac{\partial L}{\partial\gamma_j}
=
\sum_{p=0}^{P-1}D_\gamma[p,j],
\qquad
\frac{\partial L}{\partial\beta_j}
=
\sum_{p=0}^{P-1}D_\beta[p,j].
\tag{4.24}
$$

This replaces $2PE$ contended atomics with $2PE$ ordinary stores plus one
column reduction. The work is the same, but the memory system is not
serialized. The price is $8PE$ bytes of temporary storage; for $P=1024$,
$E=4096$ that is $32$ MB, which is why implementations choose $P$ (hence
`ROWS_PER_BLOCK`) as large as register pressure allows.

**Determinism.** Atomic addition is non-associative in floating point, so the
order of arrivals changes the last bits. The partial-sum buffer, reduced by a
deterministic `sum(0)`, gives reproducible gradients — valuable for debugging
and for bit-wise regression tests.

### 4.8 Backward traffic and FLOPs

Per row, the backward pass reads $X$ ($N$), $dY$ ($N$), $\gamma$ (shared, $N$),
and writes $dX$ ($N$), plus $O(1)$ for $\mu,r$. Thus

$$
\text{traffic per row}\approx4N
\ \text{elements},
\qquad
\text{FLOPs per row}\approx8N.
\tag{4.25}
$$

The parameter partials add $2PE$ stores. The arithmetic intensity is
$\approx8/(4b)=2/b$, so the backward pass is also memory-bound. Fusing the
recompute of $\hat X$ with the $dX$ computation (as above) is therefore the
right choice: it avoids an extra $MNb$ read of a saved $\hat X$.

### 4.9 Numerical stability of the backward pass

- **Use the saved $r=1/s$, not a recomputed $1/\sqrt{\sigma^{2}+\varepsilon}$.**
  Recomputing $\sigma^{2}$ from $X$ would add another rounding error and another
  pass; more importantly, it could disagree with the forward value used to
  produce $y$, breaking the exact gradient identity.
- **Keep $\mu$ and $r$ in FP32.** The factors $1/N$ and $r$ multiply every
  gradient component; FP16 storage of the statistics limits the achievable
  gradient accuracy to $\sim10^{-3}$.
- **Do the subtraction $x-\mu$ in FP32.** If $x$ is FP16 and $\mu\approx x$, the
  difference is small and must not be rounded to FP16 before squaring.
- **$\varepsilon$ placement.** With $s=\sqrt{\sigma^{2}+\varepsilon}$, a constant
  row ($\sigma^{2}=0$) gives $s=\sqrt{\varepsilon}$ and $\hat x=0$; (4.19) then
  returns a centered gradient rather than a division by zero.
- **Cancellation in $\sigma^{2}$.** If the forward used the one-pass identity
  (0.4), the backward inherits its cancellation error. The centered form is
  safer.

### 4.10 Gradient check

The formulas above can be verified against automatic differentiation with a
central finite difference,

$$
\frac{\partial L}{\partial x_i}
\approx
\frac{L(x+h e_i)-L(x-h e_i)}{2h},
\qquad
h\approx10^{-6},
\tag{4.26}
$$

and the error should decrease as $O(h^{2})$ until rounding dominates. For the
teaching size $M=8$, $N=32$, the reference implementation matches
`torch.autograd` to about $10^{-10}$ in FP64.

### 4.11 RMSNorm backward (for comparison)

For RMSNorm (0.6), let

$$
r=\sqrt{\frac1N\sum_{k=0}^{N-1}x_k^{2}+\varepsilon},
\qquad
y_j=\gamma_j\frac{x_j}{r},
\qquad
d\hat z_j=g_j\gamma_j .
\tag{4.27}
$$

Then

$$
\frac{\partial r}{\partial x_i}=\frac{x_i}{Nr},
\qquad
\frac{\partial}{\partial x_i}\left(\frac1r\right)
=
-\frac{x_i}{Nr^{3}},
\tag{4.28}
$$

and therefore

$$
\frac{\partial y_j}{\partial x_i}
=
\gamma_j\left[
\frac{\delta_{ij}}{r}
-
\frac{x_jx_i}{Nr^{3}}
\right].
\tag{4.29}
$$

Substituting into the chain rule,

$$
\boxed{\;
\frac{\partial L}{\partial x_i}
=
\frac{1}{r}\left[
d\hat z_i
-
\frac{x_i}{r^{2}}\,
\overline{d\hat z\odot x}
\right],
\qquad
\overline{d\hat z\odot x}
=
\frac1N\sum_{j=0}^{N-1}d\hat z_jx_j .
\;}
\tag{4.30}
$$

Compared with (4.19), there is **no mean-subtraction term** and the projection
is only onto $x$, not onto $\operatorname{span}\{\mathbf{1},x\}$. This is the
algebraic signature of dropping the mean.

### 4.12 Summary table

| Quantity | Formula | Reduction axis |
|---|---|---|
| $d\hat z$ | $g\odot\gamma$ | none |
| $\partial L/\partial\beta_j$ | $\sum_r g_{rj}$ | rows |
| $\partial L/\partial\gamma_j$ | $\sum_r g_{rj}\hat x_{rj}$ | rows |
| $\overline{d\hat z}$ | $\frac1N\sum_j d\hat z_j$ | features |
| $\overline{d\hat z\odot\hat x}$ | $\frac1N\sum_j d\hat z_j\hat x_j$ | features |
| $\partial L/\partial x_i$ | $r\,[d\hat z_i-\overline{d\hat z}-\hat x_i\overline{d\hat z\odot\hat x}]$ | none |

**Self-check.** Re-derive (4.16) from (4.11)–(4.14), then substitute into
(4.10) to obtain (4.19). Verify that both terms of the projection are needed by
testing the two invariances: adding a constant to $x$, and scaling $x$ by a
positive constant.


In [ ]:
def _rows_per_block(block_size: int) -> int:
    """Keep a (ROWS_PER_BLOCK x BLOCK_SIZE) tile within a sane register budget."""
    return max(1, min(16, 4096 // block_size))


@triton.jit
def _layer_norm_bwd_kernel(
    X,
    DY,
    DX,
    W,
    Mean,
    Rstd,
    DGamma,
    DBeta,
    stride_x,
    stride_dy,
    stride_dx,
    n_rows,
    n_cols: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
    ROWS_PER_BLOCK: tl.constexpr,
):
    """Method 03: tiled backward pass with partial dgamma/dbeta sums."""
    block = tl.program_id(0)
    rows = block * ROWS_PER_BLOCK + tl.arange(0, ROWS_PER_BLOCK)
    row_mask = rows < n_rows
    cols = tl.arange(0, BLOCK_SIZE)
    col_mask = cols < n_cols
    mask = row_mask[:, None] & col_mask[None, :]

    x = tl.load(
        X + rows[:, None] * stride_x + cols[None, :], mask=mask, other=0.0
    ).to(tl.float32)
    dy = tl.load(
        DY + rows[:, None] * stride_dy + cols[None, :], mask=mask, other=0.0
    ).to(tl.float32)
    mean = tl.load(Mean + rows, mask=row_mask, other=0.0)[:, None]
    rstd = tl.load(Rstd + rows, mask=row_mask, other=0.0)[:, None]

    x_hat = (x - mean) * rstd
    if W is not None:
        w = tl.load(W + cols, mask=col_mask, other=0.0).to(tl.float32)[None, :]
    else:
        w = 1.0

    dz = dy * w
    mean1 = tl.sum(dz * x_hat, axis=1)[:, None] / n_cols
    mean2 = tl.sum(dz, axis=1)[:, None] / n_cols
    dx = rstd * (dz - x_hat * mean1 - mean2)
    tl.store(DX + rows[:, None] * stride_dx + cols[None, :], dx, mask=mask)

    # Partial sums for this block of rows: (n_blocks, n_cols) buffers.
    if DGamma is not None:
        tl.store(
            DGamma + block * n_cols + cols,
            tl.sum(dy * x_hat, axis=0),
            mask=col_mask,
        )
    if DBeta is not None:
        tl.store(
            DBeta + block * n_cols + cols,
            tl.sum(dy, axis=0),
            mask=col_mask,
        )


def layer_norm_training(x, weight, bias, eps=1e-5):
    """Forward that also returns the FP32 statistics needed by backward."""
    m, n = x.shape
    x = x.contiguous()          # the kernel assumes a dense trailing dimension
    y = torch.empty_like(x)
    mean = torch.empty((m,), device=x.device, dtype=torch.float32)
    rstd = torch.empty((m,), device=x.device, dtype=torch.float32)
    block = triton.next_power_of_2(n)
    _layer_norm_fwd_kernel[(m,)](
        y,
        x,
        weight,
        bias,
        mean,
        rstd,
        x.stride(0),
        n,
        eps,
        BLOCK_SIZE=block,
    )
    return y, mean, rstd


def layer_norm_backward(dy, x, mean, rstd, weight=None, bias=None):
    """Method 03: Triton backward; returns (dx, dgamma, dbeta)."""
    n_rows, n_cols = x.shape
    x = x.contiguous()
    dy = dy.contiguous()
    dx = torch.empty_like(x)
    block = triton.next_power_of_2(n_cols)
    rows_per_block = _rows_per_block(block)
    n_blocks = triton.cdiv(n_rows, rows_per_block)

    dgamma_blocks = (
        torch.empty((n_blocks, n_cols), device=x.device, dtype=torch.float32)
        if weight is not None
        else None
    )
    dbeta_blocks = (
        torch.empty((n_blocks, n_cols), device=x.device, dtype=torch.float32)
        if bias is not None
        else None
    )

    _layer_norm_bwd_kernel[(n_blocks,)](
        x,
        dy,
        dx,
        weight,
        mean,
        rstd,
        dgamma_blocks,
        dbeta_blocks,
        x.stride(0),
        dy.stride(0),
        dx.stride(0),
        n_rows,
        n_cols,
        block,
        rows_per_block,
    )

    dgamma = (
        dgamma_blocks.sum(0).to(weight.dtype) if weight is not None else None
    )
    dbeta = dbeta_blocks.sum(0).to(bias.dtype) if bias is not None else None
    return dx, dgamma, dbeta


def layer_norm_backward_ref(dy, x, mean, rstd, weight=None, bias=None):
    """CPU/GPU reference implementing equation (4.19) directly."""
    x_hat = (x - mean[:, None]) * rstd[:, None]
    dz = dy * (weight if weight is not None else 1.0)
    m1 = (dz * x_hat).mean(dim=1, keepdim=True)
    m2 = dz.mean(dim=1, keepdim=True)
    dx = rstd[:, None] * (dz - x_hat * m1 - m2)
    dgamma = (dy * x_hat).sum(0) if weight is not None else None
    dbeta = dy.sum(0) if bias is not None else None
    return dx, dgamma, dbeta


### Pitfalls

- Two-pass mean then var can be less stable than Welford; still OK for teaching.
- Forgetting $\varepsilon$ under the square root.
- Affine $\gamma,\beta$ must match the normalized feature axis length $N$.
- Saving $\mu$ and $r_{\mathrm{std}}$ in low precision destroys the backward pass.
- Using the one-pass identity (0.4) when $\mu^2/\sigma^2$ is large.

### Self-check

1. Write $\mu,\sigma^{2},\hat x,y$ for a 4-vector by hand.
2. Prove zero mean and variance $\sigma^{2}/(\sigma^{2}+\varepsilon)$.
3. How many HBM reads of $x$ in naive vs fused?
4. What changes for RMSNorm?
5. Derive (4.8) below without looking.


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    x = torch.randn(4, 32)
    w = torch.randn(32)
    b = torch.randn(32)
    ref = torch.nn.functional.layer_norm(x, (32,), w, b)
    assert torch.allclose(naive_layer_norm_pseudo(x, w, b), ref, atol=1e-4, rtol=1e-4)
    assert torch.allclose(fused_layer_norm_pseudo(x, w, b), ref, atol=1e-4, rtol=1e-4)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(128, 256, device=DEVICE)
    w = torch.randn(256, device=DEVICE)
    b = torch.randn(256, device=DEVICE)
    y = layer_norm(x, w, b)
    ref = torch.nn.functional.layer_norm(x, (256,), w, b)
    assert torch.allclose(y, ref, atol=1e-2, rtol=1e-2)
    print("layer_norm OK")


In [ ]:
def test_layer_norm_backward_ref_matches_autograd() -> None:
    """CPU check of equation (4.19) against PyTorch autograd."""
    torch.manual_seed(0)
    x = torch.randn(8, 32, dtype=torch.float64, requires_grad=True)
    w = torch.randn(32, dtype=torch.float64, requires_grad=True)
    b = torch.randn(32, dtype=torch.float64, requires_grad=True)
    eps = 1e-5
    y = torch.nn.functional.layer_norm(x, (32,), w, b, eps=eps)
    dy = torch.randn_like(y)
    dx_ref, dw_ref, db_ref = torch.autograd.grad(y, (x, w, b), dy)

    with torch.no_grad():
        mean = x.mean(dim=1)
        var = x.var(dim=1, unbiased=False)
        rstd = 1.0 / torch.sqrt(var + eps)
        dx, dw, db = layer_norm_backward_ref(dy, x, mean, rstd, w, b)

    assert torch.allclose(dx, dx_ref, atol=1e-10, rtol=1e-10)
    assert torch.allclose(dw, dw_ref, atol=1e-10, rtol=1e-10)
    assert torch.allclose(db, db_ref, atol=1e-10, rtol=1e-10)
    print("layer-norm backward reference OK")


def test_layer_norm_training_kernel() -> None:
    """GPU check of the fused forward + tiled backward kernels."""
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(128, 256, device=DEVICE, dtype=torch.float32)
    w = torch.randn(256, device=DEVICE, dtype=torch.float32)
    b = torch.randn(256, device=DEVICE, dtype=torch.float32)
    eps = 1e-5

    y, mean, rstd = layer_norm_training(x, w, b, eps=eps)
    y_ref = torch.nn.functional.layer_norm(x, (256,), w, b, eps=eps)
    assert torch.allclose(y, y_ref, atol=1e-4, rtol=1e-4)

    dy = torch.randn_like(y)
    dx, dw, db = layer_norm_backward(dy, x, mean, rstd, w, b)

    xr = x.detach().clone().requires_grad_(True)
    wr = w.detach().clone().requires_grad_(True)
    br = b.detach().clone().requires_grad_(True)
    yr = torch.nn.functional.layer_norm(xr, (256,), wr, br, eps=eps)
    dx_ref, dw_ref, db_ref = torch.autograd.grad(yr, (xr, wr, br), dy)
    assert torch.allclose(dx, dx_ref, atol=1e-3, rtol=1e-3)
    assert torch.allclose(dw, dw_ref, atol=1e-3, rtol=1e-3)
    assert torch.allclose(db, db_ref, atol=1e-3, rtol=1e-3)
    print("layer-norm training kernel OK")


In [ ]:
def test_mask_tail() -> None:
    """GPU: N=255 is not a power of two, so the mask path is live."""
    require_gpu()
    torch.manual_seed(0)
    x = torch.randn(8, 255, device=DEVICE)
    weight = torch.randn(255, device=DEVICE)
    bias = torch.randn(255, device=DEVICE)
    ref = torch.nn.functional.layer_norm(x, (255,), weight, bias, 1e-5)
    out = layer_norm(x, weight, bias)
    assert torch.allclose(out, ref, atol=1e-4, rtol=1e-4)
    print("layer-norm N=255 mask tail OK")
